# Extract Sequential Decision Tree Results to Long Format
Convert nested JSON to flat table with paper_id, domain, and risk_score columns

In [8]:
import json
from pathlib import Path

import pandas as pd

sequential_path = Path('../output/sequential_decision_tree_all_papers/all_papers_anthropic_claude-opus-5.json')
tree_guided_path = Path('../output/all_papers_all_domains_two_prompts/tree_guided_all_papers_anthropic_claude-opus-5.json')

with sequential_path.open() as file:
    sequential_data = json.load(file)
with tree_guided_path.open() as file:
    tree_guided_data = json.load(file)

def risk_label(value):
    value = str(value).upper()
    if 'NOT APPLICABLE' in value:
        return 'Not Applicable'
    if 'HIGH' in value:
        return 'High'
    if 'MEDIUM' in value or 'MED' in value:
        return 'Medium'
    if 'LOW' in value:
        return 'Low'
    return None

sequential_rows = []
for paper_id, domains in sequential_data.items():
    for domain, result_data in domains.items():
        result = result_data.get('result') if isinstance(result_data, dict) else result_data
        risk = risk_label(result)
        if risk and domain.isdigit():
            sequential_rows.append({
                'paper_id': paper_id,
                'domain': f'domain_{domain}',
                'sequential_risk': risk,
            })

tree_guided_rows = []
for paper_file, domains in tree_guided_data['results'].items():
    for domain, result_data in domains.items():
        judgement = result_data.get('parsed_response', {}).get('final_risk_judgement')
        risk = risk_label(judgement)
        if risk:
            tree_guided_rows.append({
                'paper_id': paper_file.removesuffix('.json'),
                'domain': domain,
                'tree_guided_risk': risk,
            })

df_comparison = pd.DataFrame(sequential_rows).merge(
    pd.DataFrame(tree_guided_rows), on=['paper_id', 'domain'], how='inner'
)
df_comparison['agreement'] = (
    df_comparison['sequential_risk'] == df_comparison['tree_guided_risk']
)

if df_comparison.empty:
    print('No matching paper-domain assessments were found.')
else:
    matches = df_comparison['agreement'].sum()
    total = len(df_comparison)
    print(f'Compared {total} paper-domain assessments')
    print(f'Overall agreement: {matches / total:.1%} ({matches}/{total})')

    print('\nAgreement by domain:')
    display(
        df_comparison.groupby('domain')['agreement']
        .agg(matches='sum', total='count', agreement='mean')
        .sort_values('agreement', ascending=False)
    )

    na_rows = df_comparison[
        (df_comparison['sequential_risk'] == 'Not Applicable')
        | (df_comparison['tree_guided_risk'] == 'Not Applicable')
    ]
    na_matches = na_rows['agreement'].sum()
    print(f'\nNot Applicable comparisons: {len(na_rows)} ({na_matches} agreements)')
    display(na_rows.head(90))

    mismatches = df_comparison.loc[
        ~df_comparison['agreement'],
        ['paper_id', 'domain', 'sequential_risk', 'tree_guided_risk'],
    ]
    print(f'\nAll mismatches: {len(mismatches)}')
    display(mismatches.head(90))

Compared 328 paper-domain assessments
Overall agreement: 75.0% (246/328)

Agreement by domain:


,matches,total,agreement
domain,,,
domain_3,43,48,0.895833
domain_7,39,47,0.829787
domain_4,39,47,0.829787
domain_1,34,46,0.739130
domain_2,32,46,0.695652
domain_6,31,47,0.659574
domain_5,28,47,0.595745



Not Applicable comparisons: 48 (47 agreements)


,paper_id,domain,sequential_risk,tree_guided_risk,agreement
2,038,domain_3,Not Applicable,Not Applicable,True
9,011,domain_3,Not Applicable,Not Applicable,True
16,013,domain_3,Not Applicable,Not Applicable,True
23,031,domain_3,Not Applicable,Not Applicable,True
30,009,domain_3,Not Applicable,Not Applicable,True
37,001,domain_3,Not Applicable,Not Applicable,True
44,007,domain_3,Not Applicable,Not Applicable,True
52,016,domain_4,Not Applicable,Not Applicable,True
58,026,domain_3,Not Applicable,Not Applicable,True
66,005,domain_4,Not Applicable,Not Applicable,True



All mismatches: 82


,paper_id,domain,sequential_risk,tree_guided_risk
1,038,domain_2,Low,Medium
3,038,domain_4,Low,Medium
4,038,domain_5,Medium,Low
7,011,domain_1,High,Low
12,011,domain_6,Low,High
...,...,...,...,...
319,014,domain_6,Low,Medium
321,042,domain_1,High,Medium
325,042,domain_5,Medium,Low
326,042,domain_6,Low,Medium
